# GS 2027 · Fábrica de Corpus CitizenTech — **Bairro em Dia**

Pipeline que transforma documentos públicos do **estado de São Paulo** em corpus para ajustar (fine-tuning) um modelo de linguagem pequeno que atua como **assistente do app Bairro em Dia**.

- **Persona:** morador de bairro urbano de SP, com pouca familiaridade com a burocracia, que não sabe a qual órgão recorrer nem como acompanhar o pedido.
- **Problema:** relatos de buraco, poste apagado, lixo acumulado e barulho se perdem porque o cidadão não sabe onde e como pedir atendimento.
- **ODS:** 11 (Cidades e Comunidades Sustentáveis); secundário: 16 (Instituições Eficazes).

### Estável × volátil
| Estável (vai para o treino) | Volátil (fica no RAG) |
|---|---|
| Tom de atendente, formato de um bom relato (local, descrição, foto, protocolo) | Telefones e canais do município (ex.: 156), horários de coleta |
| Direitos do usuário, existência de ouvidoria, prazos legais da LAI e da Lei 13.460 | Prazos de atendimento por órgão, valores de multas, leis municipais |
| Quando recusar ou encaminhar; urgência → 192, 193, 199 | Nomes de secretarias e contatos que mudam de gestão |

> O notebook roda do início ao fim **sem chave de API**: os modelos são abertos (Hugging Face) e as saídas geradas ficam salvas em `saida/geracao/`.

## 0. Instalação (Colab)

In [ ]:
!pip -q install pypdf python-docx beautifulsoup4 lxml ftfy spacy sentence-transformers gradio requests pandas accelerate
!python -m spacy download pt_core_news_lg -q

## 1. Projeto e ingestão de documentos
Se `corpus_factory.py` não estiver na pasta, a célula clona o repositório `Tonelli2003/GS-PLN`. Se os arquivos estiverem numa subpasta, ajuste o `%cd`. Também dá para enviar os arquivos pelo painel lateral do Colab.

In [ ]:
from pathlib import Path
if not Path("corpus_factory.py").exists():
    !git clone https://github.com/Tonelli2003/GS-PLN
    %cd GS-PLN

assert Path("corpus_factory.py").exists(), "corpus_factory.py não encontrado: confira a pasta (%cd)."

import json, random, shutil
import pandas as pd
import corpus_factory as cf

ENTRADA, SEEDS, SAIDA = Path("dados/entrada"), Path("dados/seeds"), Path("saida")
SAIDA.mkdir(exist_ok=True)

manifesto = cf.carregar_manifesto("dados/manifesto_documentos.csv")
pd.DataFrame(manifesto)[["arquivo", "titulo", "orgao", "data", "tipo_data", "em_corpus", "link_verificado"]]

### 1.1 Download dos documentos oficiais
Cada documento tem **nome do arquivo, título, órgão emissor, link de origem e data** no `manifesto_documentos.csv`. Se algum link falhar, baixe o arquivo manualmente (PDF, TXT, DOCX ou HTML) e envie para `dados/entrada/` com o mesmo nome do manifesto.

In [ ]:
resultado = cf.baixar_documentos(manifesto, ENTRADA)
for arq, status in resultado.items():
    print(f"{arq:36s} {status}")

In [ ]:
# Opcional: texto colado diretamente (entra no corpus como documento extra)
TEXTOS_COLADOS = []   # ex.: ["Cole aqui um trecho de documento oficial ..."]

docs, ausentes = cf.ingerir(manifesto, ENTRADA, TEXTOS_COLADOS)
assert not ausentes, f"Documentos ausentes em dados/entrada/: {ausentes}"
pd.DataFrame([{"arquivo": d["arquivo"], "caracteres_brutos": len(d["texto_bruto"]),
               "em_corpus": d["em_corpus"]} for d in docs])

## 2. Limpeza e normalização (exemplo antes × depois)

In [ ]:
d0 = next(d for d in docs if d["em_corpus"] == "sim")
limpo = cf.limpar_texto(d0["texto_bruto"], reflow_linhas=d0["arquivo"].endswith(".pdf"))
norm = cf.normalizar_texto(limpo)
print("ANTES :", d0["texto_bruto"][:600].replace("\n", " ⏎ "))
print("\nDEPOIS:", norm[:600].replace("\n", " ⏎ "))

## 3. Anonimização
Regex para **CPF, e-mail, telefone** (e RG/endereço) + **NER (spaCy `pt_core_news_lg`)** para nomes de pessoas. Cada dado vira um marcador (`[CPF]`, `[EMAIL]`, `[TELEFONE]`, `[NOME]`…) e **nenhuma tabela de correspondência é guardada** (irreversível = anonimizado, não pseudonimizado).

**Exceções adotadas (não mascarar):** nomes de leis (ex.: *Lei Maria da Penha*), instituições (*Fiocruz*, *Defesa Civil*, *SAMU*, *Corpo de Bombeiros*, *Defensoria Pública*, *Ministério Público*) e termos institucionais (`TERMOS_NAO_NOME`). Os blocos de assinatura das leis (nomes de autoridades) são removidos na limpeza.

In [ ]:
nlp = cf.carregar_ner()          # None => só regras (aviso é exibido)
ficha = next(d for d in docs if d["arquivo"].startswith("ficha_atendimento"))
original = cf.normalizar_texto(ficha["texto_bruto"])
anonimizada, contagem = cf.anonimizar(original, nlp)
print("===== ANTES =====\n" + original)
print("\n===== DEPOIS =====\n" + anonimizada)
print("\nDados trocados:", dict(contagem))

## 4. Segmentação e corpus de texto puro
Trechos de **256 a 512 tokens** (tokenizador do `Qwen/Qwen2.5-0.5B-Instruct`), sem cortar frases. A ficha fictícia **não** entra nos corpus (`em_corpus = nao`).

In [ ]:
contar = cf.carregar_contador()    # AutoTokenizer do Qwen2.5-0.5B-Instruct
trechos, rel_docs, cont_pii = cf.construir_corpus_texto(docs, contar, nlp)
trechos, dup_texto = cf.deduplicar(trechos, lambda r: r["text"])

# O Código de Trânsito é muito amplo (CNH, crimes de trânsito...). Fica só o que toca o tema do app.
FILTRAR_CTB_NO_TEXTO = True
if FILTRAR_CTB_NO_TEXTO:
    antes = len(trechos)
    trechos = cf.filtrar_relevantes(trechos)
    print(f"Filtro temático do CTB: {antes} -> {len(trechos)} trechos")

total_palavras = sum(len(t["text"].split()) for t in trechos)
tokens = [contar(t["text"]) for t in trechos]
print(f"{len(trechos)} trechos | {total_palavras} palavras | tokens {min(tokens)}-{max(tokens)} | "
      f"fora da faixa 256-512: {sum(not 256 <= x <= 512 for x in tokens)} | duplicatas: {dup_texto}")
print("Dados pessoais mascarados nos documentos oficiais:", dict(cont_pii))
assert len(trechos) >= 50, "Menos de 50 trechos: use mais documentos (mín. ~15 mil palavras)."
assert total_palavras >= 15000, "Menos de ~15 mil palavras."
print("Trechos descartados por ficarem fora de 256-512 tokens:",
      sum(v["descartados_fora_da_faixa"] for v in rel_docs.values()))
tabela = pd.DataFrame(rel_docs).T
tabela["trechos_no_corpus"] = pd.Series([t["fonte"] for t in trechos]).value_counts()
tabela

In [ ]:
cf.salvar_jsonl([{k: t[k] for k in ("text", "fonte", "secao", "data", "id", "titulo", "orgao", "link")}
                 for t in trechos], SAIDA / "corpus_texto.jsonl")

## 5. Conjunto de teste (criado **antes** dos corpus de treino e não alterado)
O `dados/seeds/corpus_teste.jsonl` foi escrito à mão antes de qualquer geração. O notebook só o **copia** e registra o hash SHA-256 para provar que ele não mudou.

In [ ]:
teste = cf.ler_jsonl(SEEDS / "corpus_teste.jsonl")
shutil.copyfile(SEEDS / "corpus_teste.jsonl", SAIDA / "corpus_teste.jsonl")
tipos = pd.Series([t["tipo"] for t in teste]).value_counts()
print(len(teste), "perguntas |", tipos.to_dict(), "| SHA-256:", cf.sha256_arquivo(SEEDS / "corpus_teste.jsonl")[:16], "...")
print("Língua real no teste:", f"{sum(cf.eh_lingua_real(t['pergunta']) for t in teste) / len(teste):.0%}")
assert len(teste) >= 20 and tipos.get("fora_do_tema", 0) >= 3 and tipos.get("urgencia", 0) >= 2

## 6. Geração dos pares de perguntas, respostas e diálogos (modelo aberto, sem API)
Modelo padrão: `Qwen/Qwen2.5-3B-Instruct` (cabe numa GPU T4 do Colab). Prompts em `cf.PROMPT_QA` e `cf.PROMPT_CHAT`. As saídas brutas são salvas em `saida/geracao/*.jsonl`: com o cache presente, `GERAR_COM_LLM = False` reproduz tudo sem GPU.

In [ ]:
GERAR_COM_LLM = True
MODELO_LLM = "Qwen/Qwen2.5-3B-Instruct"
llm = cf.carregar_llm(MODELO_LLM) if GERAR_COM_LLM else None

# Do Código de Trânsito (muito amplo), só entram na geração os trechos que tocam o tema do app
trechos_gen = cf.filtrar_relevantes(trechos)
print(len(trechos_gen), "de", len(trechos), "trechos usados na geração")

# Gera em lotes e para quando houver exemplos válidos suficientes (o cache em arquivo permite retomar)
saidas_qa = cf.gerar_ate_alvo(trechos_gen, cf.PROMPT_QA, SAIDA / "geracao/raw_qa.jsonl", llm,
                              cf.montar_qa, alvo=130, n=3)
trechos_chat = [t for t in trechos_gen if t["fonte"] != "lei_9503_1997_ctb.html"]  # CTB gera diálogos fracos
saidas_chat = cf.gerar_ate_alvo(trechos_chat, cf.PROMPT_CHAT, SAIDA / "geracao/raw_chat.jsonl", llm,
                                cf.montar_chat, alvo=24)
print(len(saidas_qa), "saídas de Q&A |", len(saidas_chat), "saídas de chat")

## 7. Montagem dos corpus
- **Q&A** (formato Alpaca): cada par traz o `trecho` que o sustenta. Filtro automático de fidelidade (números e palavras da resposta precisam estar no trecho) + revisão humana (seção 9).
- **Conversacional**: 30 diálogos escritos à mão (recusa, encaminhamento, **urgência 192/193/199** e orientação com base nas leis) + diálogos gerados pelo modelo (2 trocas, a 2ª depende da 1ª). Os gerados precisam passar pela revisão humana.
- **Língua real**: ≥ 30% das perguntas do usuário reescritas com erros, caixa alta e abreviações. As respostas **não** são alteradas.

In [ ]:
# ---- Q&A
qa, desc_qa = cf.montar_qa(trechos, saidas_qa)
print("Descartes Q&A:", dict(cf.MOTIVOS))
novas = cf.aplicar_lingua_real([p["instruction"] for p in qa], fracao=0.4)
for p, n in zip(qa, novas):
    p["instruction"] = n
qa, dup_qa = cf.deduplicar(qa, lambda r: r["instruction"] + " " + r["output"])

# ---- Conversacional
gerados, desc_chat = cf.montar_chat(trechos, saidas_chat)
print("Descartes chat:", dict(cf.MOTIVOS))
for pos in (1, 3):   # 1ª e 2ª perguntas do usuário
    novas = cf.aplicar_lingua_real([d["messages"][pos]["content"] for d in gerados], fracao=0.4, semente=7 + pos)
    for d, n in zip(gerados, novas):
        d["messages"][pos]["content"] = n
seed_chat = cf.ler_jsonl(SEEDS / "chat_seed.jsonl")
chat = seed_chat + gerados
chat, dup_chat = cf.deduplicar(chat, lambda r: " ".join(m["content"] for m in r["messages"] if m["role"] == "user"))

# ---- Vazamento do teste para o treino
perguntas_teste = [t["pergunta"] for t in teste]
vaz_qa = cf.verificar_vazamento(perguntas_teste, [p["instruction"] for p in qa])
vaz_chat = cf.verificar_vazamento(perguntas_teste, [m["content"] for d in chat for m in d["messages"] if m["role"] == "user"])
print("Vazamentos Q&A:", vaz_qa, "| chat:", vaz_chat)
if vaz_qa:
    ruins = {q for _, q in vaz_qa}; qa = [p for p in qa if p["instruction"] not in ruins]
if vaz_chat:
    ruins = {q for _, q in vaz_chat}
    chat = [d for d in chat if not any(m["role"] == "user" and m["content"] in ruins for m in d["messages"])]

# ---- Quase-duplicatas (embeddings, cosseno >= 0.95)
qa, nd1 = cf.remover_quase_duplicatas(qa, lambda r: r["instruction"])
dup_qa += nd1

# ---- Língua real: checagem
users_qa = [p["instruction"] for p in qa]
users_chat = [m["content"] for d in chat for m in d["messages"] if m["role"] == "user"]
print(f"Língua real: Q&A {sum(map(cf.eh_lingua_real, users_qa)) / len(users_qa):.0%} | "
      f"chat {sum(map(cf.eh_lingua_real, users_chat)) / len(users_chat):.0%}")

# ---- Salvar (o campo interno 'categoria' não vai para o arquivo final)
recusas_chat = (sum(1 for d in chat if d.get("categoria") in cf.CATEGORIAS_RECUSA)
                + cf.conta_recusas([d for d in chat if not d.get("categoria")]))
cf.salvar_jsonl(qa, SAIDA / "corpus_qa.jsonl")
cf.salvar_jsonl([{k: v for k, v in d.items() if k != "categoria"} for d in chat], SAIDA / "corpus_chat.jsonl")

## 8. Relatório de qualidade

In [ ]:
est = [
    cf.estatisticas("corpus_texto", trechos, lambda r: r["text"], contar, dup_texto),
    cf.estatisticas("corpus_qa", qa, lambda r: r["instruction"] + "\n" + r["output"], contar, dup_qa,
                    recusas=sum("não posso" in p["output"].lower() or "procure" in p["output"].lower() for p in qa)),
    cf.estatisticas("corpus_chat", chat, lambda r: "\n".join(m["content"] for m in r["messages"] if m["role"] != "system"), contar,
                    dup_chat, recusas=recusas_chat),
    cf.estatisticas("corpus_teste", teste, lambda r: r["pergunta"], contar, 0),
]
fontes_qa = {p["fonte"] for p in qa}
print("\nDocumentos distintos no Q&A:", len(fontes_qa), sorted(fontes_qa))
print("Descartados pelo filtro de fidelidade/JSON — Q&A:", desc_qa, "| chat:", desc_chat)

# Metas do enunciado
assert len(qa) >= 100 and len(fontes_qa) >= 3, "Q&A abaixo do mínimo: gere com mais trechos/pares."
assert len(chat) >= 30 and recusas_chat >= 10, "Chat abaixo do mínimo."

# Nenhum dado pessoal pode restar nos corpus finais
for nome, regs, ext in [("texto", trechos, lambda r: r["text"]), ("qa", qa, lambda r: r["instruction"] + r["output"] + r["trecho"]),
                        ("chat", chat, lambda r: " ".join(m["content"] for m in r["messages"]))]:
    assert not cf.varrer_pii(regs, ext), f"PII encontrado em {nome}!"
print("Varredura de dados pessoais: OK")
json.dump(est, open(SAIDA / "estatisticas.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

## 8b. Auditoria final (checklist de qualidade)
Confere: contagem real de linhas dos arquivos salvos, codificação, **falsos positivos do NER** nos documentos oficiais, dados pessoais residuais (CPF, RG, telefone, e-mail, **endereço** e **nomes**) e a fatia do Código de Trânsito. As entidades listadas em 2 servem só para conferência: **não são salvas** (nada de tabela de correspondência).

In [ ]:
print("1) Linhas nos arquivos salvos:")
for nome in ["corpus_texto", "corpus_qa", "corpus_chat", "corpus_teste"]:
    n = sum(1 for _ in open(SAIDA / f"{nome}.jsonl", encoding="utf-8"))
    print(f"   {nome}.jsonl: {n}")

print("\n2) Codificação quebrada (Ã§, Ã£, �):")
bad = (cf.auditar_codificacao(trechos, lambda r: r["text"])
       + cf.auditar_codificacao(qa, lambda r: r["instruction"] + r["output"])
       + cf.auditar_codificacao(chat, lambda r: " ".join(m["content"] for m in r["messages"])))
print("   nenhuma ocorrência" if not bad else f"   {len(bad)} ocorrências: {bad[:5]}")

print("\n3) Entidades PER detectadas nos documentos oficiais (conferência; mascarada = True/False):")
for (arq, ent, masc), n in cf.auditar_ner([d for d in docs if d["em_corpus"] == "sim"], nlp).most_common(40):
    print(f"   {'MASCARADA ' if masc else 'mantida   '} {arq:32s} {ent!r} x{n}")

print("\n4) Dados pessoais residuais nos corpus finais:")
fontes = [("texto", trechos, lambda r: r["text"]),
          ("qa", qa, lambda r: r["instruction"] + " " + r["output"] + " " + r["trecho"]),
          ("chat", chat, lambda r: " ".join(m["content"] for m in r["messages"]))]
for nome, regs, ext in fontes:
    print(f"   {nome:6s} regex: {cf.varrer_pii(regs, ext)}  |  nomes (NER): {cf.varrer_nomes(regs, ext, nlp)}")

ctb = sum(t["fonte"] == "lei_9503_1997_ctb.html" for t in trechos)
print(f"\n5) Código de Trânsito no texto puro: {ctb} de {len(trechos)} trechos ({ctb / len(trechos):.0%})")

## 9. Revisão humana e conferência de fidelidade
O enunciado exige revisar **≥ 20%** dos exemplos gerados por IA e registrar em `revisao.csv`. O notebook gera o **modelo** do arquivo e uma amostra de 20 pares para conferência. **Preencha à mão** (`alterado` = sim/não e `comentário`) e informe no relatório quantos dos 20 estavam fiéis ao trecho.

In [ ]:
n_rev = cf.exportar_revisao({"corpus_qa.jsonl": len(qa), "corpus_chat.jsonl": len(chat)}, SAIDA / "revisao.csv")
print(n_rev, "linhas em saida/revisao.csv (preencher a coluna 'alterado' e 'comentário').")

rng = random.Random(5)
amostra = rng.sample(range(len(qa)), 20)
pd.DataFrame([{"linha": i + 1, "pergunta": qa[i]["instruction"], "resposta": qa[i]["output"],
               "trecho": qa[i]["trecho"][:400] + "...", "fiel (s/n)": ""} for i in amostra]
             ).to_csv(SAIDA / "amostra_fidelidade_20.csv", index=False, encoding="utf-8-sig")

## 10. Interface simples (Gradio): do upload à geração do corpus de texto
Demonstração para o vídeo: envie PDF/TXT/DOCX/HTML e/ou cole texto; a interface roda limpeza, normalização, anonimização e segmentação e mostra as estatísticas e 3 trechos.

In [ ]:
import gradio as gr
from collections import Counter

def processar(arquivos, texto_colado):
    pasta = Path("dados/upload"); pasta.mkdir(parents=True, exist_ok=True)
    manifesto_ui = []
    for a in (arquivos or []):
        destino = pasta / Path(a).name
        shutil.copyfile(a, destino)
        manifesto_ui.append({"arquivo": destino.name, "titulo": destino.stem, "orgao": "não informado",
                             "link": "", "data": cf.DATA_CONSULTA, "em_corpus": "sim"})
    docs_ui, _ = cf.ingerir(manifesto_ui, pasta, [texto_colado] if texto_colado and texto_colado.strip() else [])
    regs, rel, pii = cf.construir_corpus_texto(docs_ui, contar, nlp)
    if not regs:
        return "Nenhum texto encontrado.", ""
    tk = [contar(r["text"]) for r in regs]
    resumo = (f"{len(regs)} trechos | média {sum(tk) / len(tk):.0f} tokens | faixa {min(tk)}-{max(tk)}\n"
              f"Dados pessoais mascarados: {dict(pii)}\nPor documento: {rel}")
    exemplos = "\n\n".join(f"[{r['fonte']} | {r['secao']}]\n{r['text'][:500]}" for r in regs[:3])
    return resumo, exemplos

gr.Interface(fn=processar,
             inputs=[gr.File(file_count="multiple", file_types=[".pdf", ".txt", ".docx", ".html"], type="filepath",
                             label="Documentos (PDF, TXT, DOCX, HTML)"),
                     gr.Textbox(lines=6, label="Ou cole um texto aqui")],
             outputs=[gr.Textbox(label="Estatísticas"), gr.Textbox(label="Exemplos de trechos", lines=12)],
             title="Fábrica de Corpus · Bairro em Dia").launch(share=False)

## 11. Exportar os arquivos gerados (Colab)
Baixe `saida/` (corpus, `revisao.csv`, `estatisticas.json`, `amostra_fidelidade_20.csv` e o cache `geracao/`) e coloque na pasta do projeto antes do `git push`. O cache permite reproduzir tudo offline com `GERAR_COM_LLM = False`.

In [ ]:
!zip -qr saida.zip saida
from google.colab import files
files.download("saida.zip")